## Project: Geospatial Analysis using Google Places API

### Authors:
1 - Angel Francisco Cruz Briro

### Libraries

In [1]:
import numpy as np
import geopandas as gpd
from shapely.geometry import Point, box
import folium

### 1 - Generating the Spatial Grid

To begin the analysis, we first need to identify all the parks within the Merida metropolitan area. Since the Google Places API limits results to a specific radius from a central point, we will use the GeoPandas library to automatically generate a spatial grid. This will allow us to iterate over the map and ensure total coverage of the study area.

In [2]:
city_bbox = [-89.74489050365537, 20.880264095990857, -89.51284567363703, 21.08566549700958]
radio_busqueda = 1700 

In [3]:
def generar_centros_hexagonales(bbox_latlon, radio_metros, crs_utm="EPSG:32616"):
    bbox_geom = box(*bbox_latlon)
    gdf_bbox = gpd.GeoDataFrame({'geometry': [bbox_geom]}, crs="EPSG:4326")
    gdf_bbox_utm = gdf_bbox.to_crs(crs_utm)
    minx, miny, maxx, maxy = gdf_bbox_utm.total_bounds
    
    dx = radio_metros * np.sqrt(3)
    dy = radio_metros * 1.5
    centros_utm = []
    
    row = 0
    y = miny
    while y <= maxy + dy:
        x_offset = (dx / 2.0) if (row % 2 != 0) else 0.0
        x = minx + x_offset
        while x <= maxx + dx:
            centros_utm.append(Point(x, y))
            x += dx
        y += dy
        row += 1
        
    gdf_puntos_utm = gpd.GeoDataFrame(geometry=centros_utm, crs=crs_utm)
    gdf_puntos_recortados = gpd.clip(gdf_puntos_utm, gdf_bbox_utm)
    
    return gdf_puntos_recortados.to_crs("EPSG:4326")

In [4]:
puntos_api = generar_centros_hexagonales(city_bbox, radio_busqueda)

In [6]:
centro_lat = (city_bbox[1] + city_bbox[3]) / 2
centro_lon = (city_bbox[0] + city_bbox[2]) / 2
map = folium.Map(location=[centro_lat, centro_lon], zoom_start=11)

folium.Rectangle(
    bounds=[[city_bbox[1], city_bbox[0]], [city_bbox[3], city_bbox[2]]],
    color="red", fill=False, weight=2, tooltip="Zona Límite"
).add_to(map)

for index, row in puntos_api.iterrows():
    lat = row.geometry.y
    lon = row.geometry.x
    
    folium.Circle(
        location=[lat, lon],
        radius=radio_busqueda, 
        color='blue',
        weight=1,
        fill=True,
        fill_color='blue',
        fill_opacity=0.15,
        tooltip=f"Lat: {lat:.4f}, Lon: {lon:.4f}"
    ).add_to(map)
    
map